In [57]:
# 1. Importar bibliotecas
import os
import pandas as pd
import sqlite3

In [58]:
# 2. Garantir que a pasta data existe
os.makedirs("../data", exist_ok=True)

In [59]:
# 3. Criar (ou abrir) o banco de dados
db_path = "../data/nubank.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"Banco criado em: {db_path}")

Banco criado em: ../data/nubank.db


In [60]:
# 4. Ler todos os CSVs da pasta processed e salvar como tabelas
processed_path = "../data/processed"

for file in os.listdir(processed_path):
    if file.endswith(".csv"):
        table_name = file.replace(".csv", "")
        df = pd.read_csv(os.path.join(processed_path, file), sep=",", encoding="utf-8-sig")
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Tabela {table_name} criada com {len(df)} registros.")

Tabela dados_nubank_extraidos criada com 125 registros.
Tabela reclamacoes_com_clusters criada com 122 registros.
Tabela reclamacoes_com_insights criada com 122 registros.


In [61]:
# 5. Listar tabelas criadas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [t[0] for t in cursor.fetchall()]
print("Tabelas no banco:", tables)

Tabelas no banco: ['dados_nubank_extraidos', 'reclamacoes_com_clusters', 'reclamacoes_com_insights']


In [62]:
# 6. Mostrar estrutura de cada tabela
for table in tables:
    cursor.execute(f"PRAGMA table_info({table});")
    print(f"\nEstrutura da tabela {table}:")
    for col in cursor.fetchall():
        print(col)


Estrutura da tabela dados_nubank_extraidos:
(0, 'Reclamacao', 'TEXT', 0, None, 0)

Estrutura da tabela reclamacoes_com_clusters:
(0, 'Reclamacao', 'TEXT', 0, None, 0)
(1, 'Categoria', 'TEXT', 0, None, 0)
(2, 'Cluster', 'INTEGER', 0, None, 0)
(3, 'Comprimento', 'INTEGER', 0, None, 0)

Estrutura da tabela reclamacoes_com_insights:
(0, 'Reclamacao', 'TEXT', 0, None, 0)
(1, 'Categoria_Primaria', 'TEXT', 0, None, 0)
(2, 'Severidade_Estimada', 'TEXT', 0, None, 0)
(3, 'Urgencia', 'INTEGER', 0, None, 0)
(4, 'Perda_Financeira', 'INTEGER', 0, None, 0)
(5, 'Problema_Funcional', 'INTEGER', 0, None, 0)
(6, 'Problema_Seguranca', 'INTEGER', 0, None, 0)
(7, 'Cluster', 'INTEGER', 0, None, 0)


In [63]:
# 7. Contar registros em cada tabela
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    count = cursor.fetchone()[0]
    print(f"Tabela {table} tem {count} registros.")

Tabela dados_nubank_extraidos tem 125 registros.
Tabela reclamacoes_com_clusters tem 122 registros.
Tabela reclamacoes_com_insights tem 122 registros.


In [64]:
# 8. Fazer uma consulta simples em uma tabela
cursor.execute("SELECT * FROM dados_nubank_extraidos LIMIT 5;")
print(cursor.fetchall())

[('Nubank falha ao contestar transação via PIX e cliente perde R$ 200,00 em [Editado pelo Reclame Aqui]',), ('249,93',), ('Bloqueio e Desativação de Conta Nubank sem Permissão ou Motivo',), ('Banco não libera crédito nem aumenta limite mesmo com pagamento em dia e movimentação alta',), ('Não consigo acessar minha conta após troca de aparelho celular, impactando pagamentos.',)]


In [65]:
# 9. Fechar a conexão quando terminar
conn.close()